# 10장. 분류 분석으로 주문 취소 여부 예측하기

이 노트북은 `book/chapters/ch10_llm_code_generation.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 온라인 쇼핑몰 데이터를 사용해 **주문 취소 여부(is_cancelled)** 를 예측하는 이진 분류 모델링 흐름을 이해하는 것입니다.

주의: 현재 강의안 파일명에는 `llm_code_generation`이 포함되어 있지만, 실제 본문 내용은 분류 분석입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 이 노트북은 5장에서 만든 `data/processed/*_clean.csv` 파일을 사용합니다.
- 전처리 파일이 없다면 먼저 `python scripts/preprocess_data.py`를 실행하세요.
- 분류 분석 결과는 `reports/` 폴더에 저장합니다.
- 분류 모델은 accuracy만 보지 않고 precision, recall, f1-score를 함께 봅니다.
- `order_status`는 정답을 만드는 컬럼이므로 입력값으로 사용하지 않습니다. 이것은 데이터 누수입니다.


## 1. 분류 분석이란 무엇인가

분류 분석은 여러 범주 중 하나를 예측하는 머신러닝 문제입니다. 회귀 분석이 주문 금액처럼 숫자를 예측한다면, 분류 분석은 주문이 취소될지 아닌지처럼 상태를 예측합니다.

이번 장에서는 주문 상태가 `cancelled`이면 1, 그렇지 않으면 0인 `is_cancelled` 타깃을 만들고, 주문 취소 여부를 예측합니다.

| 구분 | 설명 | 이번 실습 예시 |
|---|---|---|
| 입력값(feature) | 예측에 사용할 정보 | 나이, 상품 수, 주문 금액, 결제수단, 도시 |
| 예측 대상(target) | 모델이 예측할 범주 | 주문 취소 여부 `is_cancelled` |
| 학습 데이터(train) | 모델이 패턴을 배우는 데이터 | 전체 데이터의 80% |
| 테스트 데이터(test) | 학습하지 않은 데이터로 성능 확인 | 전체 데이터의 20% |
| 평가 지표(metric) | 예측이 얼마나 맞는지 확인 | accuracy, precision, recall, f1-score |


## 2. 분류 평가 지표 읽기

취소 주문처럼 비율이 낮을 수 있는 대상을 예측할 때는 accuracy 하나만 보면 위험합니다. 대부분의 주문을 취소 아님으로 예측해도 accuracy가 높아 보일 수 있기 때문입니다.

| 지표 | 의미 | 해석할 때 주의할 점 |
|---|---|---|
| accuracy | 전체 예측 중 맞춘 비율 | 클래스 불균형이 있으면 높게 보일 수 있음 |
| precision | 취소라고 예측한 것 중 실제 취소 비율 | 잘못된 취소 예측을 줄이고 싶을 때 중요 |
| recall | 실제 취소 중 모델이 찾아낸 비율 | 취소 주문을 놓치지 않는 것이 중요할 때 사용 |
| f1-score | precision과 recall의 균형 | 두 지표를 함께 보고 싶을 때 사용 |
| confusion matrix | 예측 결과를 2×2 표로 확인 | 어떤 오류가 많은지 확인 |


## 3. 패키지와 경로 설정

scikit-learn을 사용해 Logistic Regression과 Random Forest 분류 모델을 학습합니다.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


## 4. 전처리 데이터 불러오기

10장은 5장에서 저장한 전처리 데이터를 사용합니다. 파일이 없다면 터미널에서 먼저 아래 명령을 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [ ]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('전처리 파일이 없습니다. 먼저 python scripts/preprocess_data.py 를 실행하세요.')

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
if 'signup_date' in customers.columns:
    customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

print('customers:', customers.shape, list(customers.columns))
print('orders:', orders.shape, list(orders.columns))
print('order_items:', order_items.shape, list(order_items.columns))


## 5. 타깃 컬럼 만들기

주문 상태가 `cancelled`이면 1, 아니면 0인 `is_cancelled`를 만듭니다. 이 컬럼이 모델의 정답입니다.

중요: `order_status`는 정답을 만드는 데 사용했으므로 입력값으로 사용하면 안 됩니다.


In [ ]:
display(orders['order_status'].value_counts(dropna=False))

orders['is_cancelled'] = (orders['order_status'] == 'cancelled').astype(int)

target_distribution = orders['is_cancelled'].value_counts(dropna=False).sort_index().reset_index()
target_distribution.columns = ['is_cancelled', 'count']
target_distribution['ratio'] = (
    target_distribution['count'] / target_distribution['count'].sum()
).round(4)

target_distribution.to_csv(REPORT_DIR / 'ch10_target_distribution.csv', index=False, encoding='utf-8-sig')
target_distribution


타깃 비율은 매우 중요합니다. 취소 주문이 적은 데이터에서는 모든 주문을 `취소 아님`으로 예측해도 accuracy가 높게 나올 수 있습니다. 그래서 precision, recall, f1-score도 함께 봐야 합니다.


## 6. 주문 단위 특징 만들기

주문 상세 데이터는 한 주문 안에 여러 상품이 들어 있을 수 있으므로 `order_id` 기준으로 집계합니다. 주문별 상품 개수, 총 수량, 주문 금액을 만듭니다.


In [ ]:
if 'line_total' not in order_items.columns:
    order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

order_item_features = (
    order_items
    .groupby('order_id', as_index=False)
    .agg(
        item_count=('product_id', 'count'),
        total_quantity=('quantity', 'sum'),
        order_amount=('line_total', 'sum'),
    )
)

order_item_features.head()


## 7. 주문 데이터와 고객 데이터 연결하기

주문 데이터에 주문 상세 특징과 고객 정보를 붙입니다. 병합 후에는 행 수와 누락값을 확인합니다.


In [ ]:
model_data = orders.merge(
    order_item_features,
    on='order_id',
    how='left',
)

model_data = model_data.merge(
    customers,
    on='customer_id',
    how='left',
)

for col in ['item_count', 'total_quantity', 'order_amount']:
    if col in model_data.columns:
        model_data[col] = model_data[col].fillna(0)

model_data['order_month'] = model_data['order_date'].dt.month
model_data['order_dayofweek'] = model_data['order_date'].dt.dayofweek

if 'signup_date' in model_data.columns:
    model_data['days_since_signup'] = (
        model_data['order_date'] - model_data['signup_date']
    ).dt.days
    model_data['days_since_signup'] = model_data['days_since_signup'].fillna(
        model_data['days_since_signup'].median()
    )

print('orders:', orders.shape)
print('model_data:', model_data.shape)
display(model_data[['item_count', 'total_quantity', 'order_amount']].isna().sum())
model_data.head()


## 8. 입력값과 정답 나누기

이번 실습에서는 존재하는 컬럼만 입력값으로 사용합니다. `order_status`와 `is_cancelled`는 데이터 누수 위험 컬럼이므로 feature에서 제외합니다.


In [ ]:
candidate_numeric_features = [
    'age',
    'item_count',
    'total_quantity',
    'order_amount',
    'order_month',
    'order_dayofweek',
    'days_since_signup',
]

candidate_categorical_features = [
    'gender',
    'city',
    'payment_method',
]

numeric_features = [col for col in candidate_numeric_features if col in model_data.columns]
categorical_features = [col for col in candidate_categorical_features if col in model_data.columns]
features = numeric_features + categorical_features

leakage_columns = ['order_status', 'is_cancelled']
leakage_found = [col for col in leakage_columns if col in features]
if leakage_found:
    raise ValueError(f'데이터 누수 위험 컬럼이 입력값에 포함되었습니다: {leakage_found}')

X = model_data[features].copy()
y = model_data['is_cancelled'].copy()

print('numeric_features:', numeric_features)
print('categorical_features:', categorical_features)
print('X:', X.shape)
print('y:', y.shape)
display(X.isna().sum())


## 9. 학습 데이터와 테스트 데이터 나누기

분류 문제에서는 학습 데이터와 테스트 데이터의 타깃 비율이 비슷하게 유지되도록 `stratify=y`를 사용하는 것이 좋습니다.


In [ ]:
stratify = y if y.nunique() > 1 and y.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=stratify,
)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('train target ratio')
display(y_train.value_counts(normalize=True).round(3))
print('test target ratio')
display(y_test.value_counts(normalize=True).round(3))


## 10. 숫자형/범주형 컬럼 전처리 파이프라인 만들기

숫자형 컬럼은 중앙값 대체와 표준화를 적용하고, 범주형 컬럼은 최빈값 대체와 원-핫 인코딩을 적용합니다.


In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', make_one_hot_encoder()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)


## 11. Logistic Regression 기준 모델 만들기

Logistic Regression은 이름에 Regression이 들어가지만 분류 모델입니다. 여기서는 `class_weight='balanced'`를 사용해 소수 클래스에 대한 가중치를 조정합니다.


In [ ]:
logistic_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced')),
])

logistic_model.fit(X_train, y_train)
y_pred_logistic = logistic_model.predict(X_test)

logistic_metrics = {
    'accuracy': accuracy_score(y_test, y_pred_logistic),
    'precision': precision_score(y_test, y_pred_logistic, zero_division=0),
    'recall': recall_score(y_test, y_pred_logistic, zero_division=0),
    'f1': f1_score(y_test, y_pred_logistic, zero_division=0),
}

logistic_metrics


## 12. Random Forest 비교 모델 만들기

Random Forest는 여러 개의 의사결정나무를 사용해 분류하는 모델입니다. 복잡한 패턴을 잡을 수 있지만, 해석은 더 어려울 수 있습니다.


In [ ]:
rf_preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

rf_model = Pipeline(steps=[
    ('preprocessor', rf_preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight='balanced',
    )),
])

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

rf_metrics = {
    'accuracy': accuracy_score(y_test, y_pred_rf),
    'precision': precision_score(y_test, y_pred_rf, zero_division=0),
    'recall': recall_score(y_test, y_pred_rf, zero_division=0),
    'f1': f1_score(y_test, y_pred_rf, zero_division=0),
}

rf_metrics


## 13. 모델 비교 결과 저장

모델 비교에서는 단순히 수치가 높은 모델을 고르는 것이 아니라, 어떤 지표가 중요한지 먼저 정해야 합니다. 취소 주문을 놓치지 않는 것이 중요하면 recall이 중요할 수 있습니다.


In [ ]:
model_comparison = pd.DataFrame([
    {'model': 'Logistic Regression', **logistic_metrics},
    {'model': 'Random Forest', **rf_metrics},
]).sort_values('f1', ascending=False)

model_comparison.to_csv(REPORT_DIR / 'ch10_classification_model_comparison.csv', index=False, encoding='utf-8-sig')
model_comparison


## 14. 혼동행렬과 상세 리포트 확인

혼동행렬은 실제값과 예측값의 조합을 보여 줍니다. 어떤 오류가 많은지 확인할 수 있습니다.


In [ ]:
best_model_name = model_comparison.iloc[0]['model']
best_pred = y_pred_rf if best_model_name == 'Random Forest' else y_pred_logistic
best_model = rf_model if best_model_name == 'Random Forest' else logistic_model

confusion_df = pd.DataFrame(
    confusion_matrix(y_test, best_pred, labels=[0, 1]),
    index=['actual_not_cancelled', 'actual_cancelled'],
    columns=['pred_not_cancelled', 'pred_cancelled'],
)

report_df = pd.DataFrame(
    classification_report(y_test, best_pred, zero_division=0, output_dict=True)
).T.reset_index().rename(columns={'index': 'label'})

confusion_df.to_csv(REPORT_DIR / 'ch10_confusion_matrix.csv', encoding='utf-8-sig')
report_df.to_csv(REPORT_DIR / 'ch10_classification_report.csv', index=False, encoding='utf-8-sig')

display(confusion_df)
display(report_df)


혼동행렬은 다음처럼 읽습니다.

| 구분 | 의미 |
|---|---|
| True Negative | 실제 취소 아님을 취소 아님으로 예측 |
| False Positive | 실제 취소 아님을 취소로 잘못 예측 |
| False Negative | 실제 취소를 취소 아님으로 잘못 예측 |
| True Positive | 실제 취소를 취소로 예측 |


## 15. 예측 확률과 임계값 조정

분류 모델은 보통 예측 확률을 제공합니다. 기본적으로는 확률이 0.5 이상이면 취소로 예측하지만, 상황에 따라 임계값을 조정할 수 있습니다.


In [ ]:
y_proba = best_model.predict_proba(X_test)[:, 1]

threshold_rows = []
for threshold in [0.2, 0.3, 0.4, 0.5, 0.6]:
    y_pred_threshold = (y_proba >= threshold).astype(int)
    threshold_rows.append({
        'threshold': threshold,
        'accuracy': accuracy_score(y_test, y_pred_threshold),
        'precision': precision_score(y_test, y_pred_threshold, zero_division=0),
        'recall': recall_score(y_test, y_pred_threshold, zero_division=0),
        'f1': f1_score(y_test, y_pred_threshold, zero_division=0),
    })

threshold_metrics = pd.DataFrame(threshold_rows)
threshold_metrics.to_csv(REPORT_DIR / 'ch10_threshold_metrics.csv', index=False, encoding='utf-8-sig')
threshold_metrics


임계값을 낮추면 더 많은 주문을 취소 위험으로 잡아낼 수 있어 recall이 올라갈 수 있습니다. 대신 실제로는 취소되지 않을 주문까지 취소 위험으로 예측해 precision이 낮아질 수 있습니다.


## 16. 예측 결과 저장

테스트 데이터의 실제값, 예측값, 취소 확률을 저장합니다. 이후 오분류 사례를 분석할 때 사용할 수 있습니다.


In [ ]:
prediction_result = X_test.copy()
prediction_result['actual_is_cancelled'] = y_test.values
prediction_result['predicted_is_cancelled'] = best_pred
prediction_result['cancel_probability'] = y_proba
prediction_result['model'] = best_model_name

model_data.to_csv(REPORT_DIR / 'ch10_classification_model_data.csv', index=False, encoding='utf-8-sig')
prediction_result.to_csv(REPORT_DIR / 'ch10_classification_predictions.csv', index=False, encoding='utf-8-sig')

prediction_result.sort_values('cancel_probability', ascending=False).head(10)


## 17. LLM 코드 검토 체크리스트

LLM이 만든 분류 코드는 오류 없이 실행되어도 데이터 누수, 잘못된 타깃, 부적절한 평가 지표 문제가 있을 수 있습니다. 아래 체크리스트로 검토합니다.


In [ ]:
classification_checklist = pd.DataFrame({
    'check_item': [
        'is_cancelled가 올바르게 만들어졌는가?',
        'order_status를 입력값으로 사용하지 않았는가?',
        'train_test_split에 stratify=y를 사용했는가?',
        '학습/테스트 데이터의 클래스 비율을 확인했는가?',
        '범주형 컬럼을 OneHotEncoder 등으로 처리했는가?',
        '결측치 처리가 학습 파이프라인 안에서 이루어졌는가?',
        'accuracy 외 precision, recall, f1-score를 함께 확인했는가?',
        '모델 결과를 취소 원인으로 단정하지 않았는가?',
    ],
    'status': ['□'] * 8,
})

classification_checklist.to_csv(REPORT_DIR / 'ch10_classification_checklist.csv', index=False, encoding='utf-8-sig')
classification_checklist


## 18. 분류 분석 요약 보고서 저장

모델링 데이터 개요, 타깃 분포, 모델 비교 결과, 혼동행렬, 임계값 비교, 체크리스트를 Markdown 보고서로 저장합니다.


In [ ]:
summary_text = f'''# Chapter 10 분류 분석 요약 보고서

## 1. 분석 목적

온라인 쇼핑몰 주문 데이터를 사용해 주문 취소 여부(is_cancelled)를 예측하는 이진 분류 모델을 만들었습니다.

## 2. 모델링 데이터 개요

- 행 수: {model_data.shape[0]}
- 열 수: {model_data.shape[1]}
- 예측 대상: is_cancelled
- 입력값: {', '.join(features)}

## 3. 타깃 클래스 분포

```text
{target_distribution.to_string(index=False)}
```

## 4. 모델 비교 결과

```text
{model_comparison.to_string(index=False)}
```

## 5. 혼동행렬

```text
{confusion_df.to_string()}
```

## 6. 임계값별 성능 비교

```text
{threshold_metrics.to_string(index=False)}
```

## 7. LLM 코드 검토 체크리스트

```text
{classification_checklist.to_string(index=False)}
```

## 8. 해석 시 주의사항

- 취소 주문 비율이 낮으면 accuracy만으로 모델을 평가하면 위험합니다.
- 취소 주문을 놓치지 않는 것이 중요하면 recall을 함께 확인해야 합니다.
- 취소 위험 알림의 정확도를 높이고 싶다면 precision을 함께 확인해야 합니다.
- 임계값을 낮추면 recall이 올라갈 수 있지만 precision은 낮아질 수 있습니다.
- 모델은 취소 여부와 입력값 사이의 패턴을 학습한 것이며, 취소 원인을 증명하지는 않습니다.
- order_status를 입력값으로 사용하면 정답을 미리 알려 주는 데이터 누수가 발생합니다.
'''

summary_path = REPORT_DIR / 'ch10_classification_summary.md'
summary_path.write_text(summary_text, encoding='utf-8')
print('분류 분석 보고서 저장 완료:', summary_path)


## 19. 소스 모듈로 전체 분류 분석 실행

위에서 단계별로 실행한 분류 분석은 `src/classification.py`에 함수로 정리되어 있습니다. 전체 파이프라인을 한 번에 실행할 수 있습니다.


In [ ]:
from src.classification import run_classification_analysis

classification_result = run_classification_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    random_state=42,
)

classification_result['model_comparison']


## 20. 스크립트로 한 번에 실행하기

터미널에서 프로젝트 루트 기준으로 아래 명령을 실행하면 10장 분류 분석 전체가 자동으로 실행됩니다.

```bash
python scripts/run_classification_analysis.py
```


## 21. LLM에게 분류 분석 코드를 요청하는 프롬프트

LLM에게 모델링 코드를 요청할 때는 데이터 구조, 목표, 금지할 데이터 누수 조건, 평가 지표를 함께 제공합니다.

```text
온라인 쇼핑몰 주문 데이터로 주문 취소 여부를 예측하는 이진 분류 모델을 만들고 싶습니다.

데이터 구조:
- orders: order_id, customer_id, order_date, payment_method, order_status
- customers: customer_id, gender, age, city, signup_date
- order_items: order_id, product_id, quantity, unit_price, line_total

목표:
- order_status가 cancelled이면 1, 그렇지 않으면 0인 is_cancelled를 만듭니다.
- order_status는 입력 feature로 사용하지 않습니다.
- Logistic Regression과 RandomForestClassifier를 비교합니다.
- train/test split에는 stratify=y를 사용합니다.
- accuracy, precision, recall, f1-score, confusion matrix를 출력합니다.

주의:
- 데이터에 없는 컬럼명을 만들지 마세요.
- 데이터 누수가 생기지 않도록 설명해 주세요.
- 각 단계에 초보자용 주석을 포함해 주세요.
```


## 22. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 입력값에서 `order_amount`를 제외했을 때 성능이 어떻게 달라지는지 비교하세요.
2. RandomForestClassifier의 `n_estimators` 값을 50, 100, 300으로 바꿔 비교하세요.
3. 임계값을 0.2부터 0.8까지 바꿔 precision과 recall 변화를 확인하세요.
4. False Negative, 즉 실제 취소인데 취소 아님으로 예측한 사례를 찾아보세요.
5. LLM에게 분류 분석 코드를 작성하게 한 뒤 데이터 누수 여부를 검토하세요.


In [ ]:
# 과제 1. order_amount를 제외한 feature 목록으로 다시 모델을 학습해 보세요.


In [ ]:
# 과제 2. RandomForestClassifier의 n_estimators 값을 바꿔 성능을 비교해 보세요.


## 23. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 분류 분석과 회귀 분석의 차이
- 주문 취소 여부 `is_cancelled` 타깃 생성
- 주문 상세 데이터를 주문 단위 특징으로 요약
- 주문, 고객, 주문 상세 데이터 병합
- 데이터 누수 컬럼 제외
- `stratify=y`를 사용한 train/test split
- 숫자형/범주형 전처리 파이프라인 구성
- Logistic Regression과 Random Forest 비교
- accuracy, precision, recall, f1-score, confusion matrix 해석
- 예측 확률과 임계값 조정
- LLM 코드 검토 체크리스트 작성
- `src/classification.py`와 `scripts/run_classification_analysis.py`로 재현 가능한 분류 분석 구성

다음 장부터는 LLM을 데이터 분석 과정에 더 본격적으로 연결합니다.
